# Stage 2 — Character Recognition with CRNN + CTC

This notebook trains a **CRNN (CNN + bidirectional LSTM) with CTC loss** to read the 7-character plate string from a rectified plate crop. We chose CRNN+CTC because:

- it trains on the **plate string only** (CCPD has this; no per-character boxes needed),
- it tolerates residual tilt and spacing variation (no fixed-position assumption),
- it's the recognition stage used by LPDet and other CCPD-targeted projects — i.e. it's known to work on this dataset.

**Pipeline of this notebook**

1. Parse CCPD filenames, rectify each plate to 48×168 using the four ground-truth corners.
2. Build a stratified subset and a frozen test split (mirroring the pose notebook's protocol).
3. Define the character vocabulary (67 classes + CTC blank = 68 outputs).
4. Implement the CRNN, CTC training loop, and greedy decoder.
5. Run 4-fold stratified cross-validation; report mean ± std (full-plate accuracy and per-position).
6. Retrain on the full pool; evaluate once on the frozen test set.

**Important note on the input.** During training we use **ground-truth** corners from CCPD's filename. At inference time, the corners come from your YOLO-pose model. So the recognizer's measured accuracy here is an *upper bound* — real end-to-end accuracy will be slightly lower because pose corners are imperfect. To estimate that, evaluate the trained CRNN on crops rectified from **predicted** corners (we include a cell for this at the end).

## 1. Configuration & imports

In [ ]:
import os, re, math, random, shutil, json, pickle, time
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

random.seed(0); np.random.seed(0); torch.manual_seed(0)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

# ── CONFIGURE THESE ───────────────────────────────────────────────────────────
DATASET_ROOT = Path("Dataset")           # your CCPD root
WORK_ROOT    = Path("crnn_plate")        # where we cache rectified crops
CROP_W, CROP_H = 168, 48                  # CRNN input size (LPDet convention)
IMG_W, IMG_H = 720, 1160                  # original CCPD image size

TEST_FRAC      = 0.15
N_FOLDS        = 4
MAX_IMAGES     = 8000     # stratified subset cap (None for everything)
MIN_PER_SUBSET = 200

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)
if DEVICE == "cuda":
    print("       ", torch.cuda.get_device_name(0))
print("CCPD root:", DATASET_ROOT.resolve(), "| exists:", DATASET_ROOT.exists())

## 2. CCPD parser + corner-ordering (reused from the pose notebook)

We reuse the same parser and corner-ordering logic — they're verified to be correct from the pose-training pipeline. The only difference here is what we do with the parsed result: instead of writing YOLO labels, we rectify the plate to a fixed-size crop.

In [ ]:
PROVINCES = ['皖','沪','津','渝','冀','晋','蒙','辽','吉','黑','苏','浙','京',
             '闽','赣','鲁','豫','鄂','湘','粤','桂','琼','川','贵','云','藏',
             '陕','甘','青','宁','新','警','学','O']            # 34
ALPHABETS = ['A','B','C','D','E','F','G','H','J','K','L','M','N',
             'P','Q','R','S','T','U','V','W','X','Y','Z','O']   # 25
ADS = ['A','B','C','D','E','F','G','H','J','K','L','M','N',
       'P','Q','R','S','T','U','V','W','X','Y','Z',
       '0','1','2','3','4','5','6','7','8','9','O']             # 35

def parse_ccpd_filename(path):
    name = Path(path).stem
    f = name.split('-')
    if len(f) != 7: return None
    f_area, f_tilt, f_bbox, f_verts, f_label, f_bright, f_blur = f
    try:
        (x1,y1),(x2,y2) = [tuple(map(int,p.split('&'))) for p in f_bbox.split('_')]
        verts = [tuple(map(int,p.split('&'))) for p in f_verts.split('_')]
        if len(verts) != 4: return None
        idxs = list(map(int, f_label.split('_')))
        if len(idxs) != 7: return None
        plate = PROVINCES[idxs[0]] + ALPHABETS[idxs[1]] + ''.join(ADS[i] for i in idxs[2:])
    except (ValueError, IndexError):
        return None
    return {"path": str(path), "bbox": (x1,y1,x2,y2), "vertices": verts, "plate": plate}

def order_corners(pts):
    '''Return 4 points ordered as TL, TR, BR, BL (robust to input order).'''
    pts = np.asarray(pts, dtype=np.float32)
    s = pts.sum(axis=1); d = np.diff(pts, axis=1).ravel()
    return np.stack([pts[np.argmin(s)], pts[np.argmin(d)],
                     pts[np.argmax(s)], pts[np.argmax(d)]])

# sanity
_t = "025-95_113-154&383_386&473-386&473_177&454_154&383_363&402-0_0_22_27_27_33_16-95-9.jpg"
print(parse_ccpd_filename(_t)["plate"], "→", order_corners(parse_ccpd_filename(_t)["vertices"]).tolist())

## 3. Character vocabulary for CRNN+CTC

CTC needs one flat list of all possible output characters plus a special **blank** symbol (index 0 by convention). The vocabulary is the union of every character that can ever appear on a plate. That's:

- 31 province characters (excluding 警, 学 which are special-purpose plates we may not see, and the 'O' sentinel which shouldn't appear on real plates),
- 24 real Latin letters (no I, no O),
- 10 digits.

We keep 警 and 学 in the vocabulary for completeness — if your sample contains police or learner plates, the model can still output them. Sentinel 'O' is excluded since real plates never have one.

In [ ]:
# Build the character set: union across all 7 positions of all decoded plates
# We use a fixed, deterministic ordering so token indices are reproducible.

REAL_PROVINCES = [p for p in PROVINCES if p != 'O']           # 33 (incl. 警, 学)
REAL_LETTERS   = [a for a in ALPHABETS if a != 'O']           # 24
REAL_DIGITS    = [d for d in ADS if d.isdigit()]              # 10

# CTC convention: index 0 = blank
VOCAB = ['<blank>'] + REAL_PROVINCES + REAL_LETTERS + REAL_DIGITS
CHAR2IDX = {c: i for i, c in enumerate(VOCAB)}
IDX2CHAR = {i: c for c, i in CHAR2IDX.items()}
N_CLASSES = len(VOCAB)
BLANK_IDX = 0

print(f"Vocabulary size (incl. blank): {N_CLASSES}")
print(f"  provinces: {len(REAL_PROVINCES)} ({REAL_PROVINCES[:5]}…)")
print(f"  letters  : {len(REAL_LETTERS)}  ({REAL_LETTERS[:5]}…)")
print(f"  digits   : {len(REAL_DIGITS)}  ({REAL_DIGITS})")

def encode(plate: str):
    '''Encode a plate string to a list of vocab indices.'''
    return [CHAR2IDX[c] for c in plate]

def decode_indices(idxs):
    '''Decode a list of vocab indices (no CTC collapsing). For debugging.'''
    return ''.join(IDX2CHAR[i] for i in idxs)

# sanity
_p = parse_ccpd_filename(_t)["plate"]
_e = encode(_p)
print(f"{_p}  → encode: {_e}  → decode: {decode_indices(_e)}")

## 4. Rectify plates and cache them as 48×168 crops

For each CCPD image, we apply a perspective warp from the four ground-truth corners to a fixed-size rectangle. We cache the resulting crops to disk so dataloading is fast during training (no re-warping every epoch).

Caching strategy: filenames are `<stem>.jpg` in `crnn_plate/crops/`. The corresponding plate label is in `crnn_plate/labels.csv` (path → plate string + CCPD subset).

In [ ]:
def discover_subset(p: Path) -> str:
    try:
        rel = p.relative_to(DATASET_ROOT); parts = rel.parts
        return parts[0] if len(parts) > 1 else "unknown"
    except ValueError:
        return "unknown"

def rectify_plate_bgr(img_bgr, vertices, out_w=CROP_W, out_h=CROP_H):
    '''Warp the plate quad to a fixed-size upright rectangle. Returns HxWx3 uint8 BGR.'''
    corners = order_corners(vertices).astype(np.float32)
    dst = np.array([[0,0],[out_w-1,0],[out_w-1,out_h-1],[0,out_h-1]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(corners, dst)
    return cv2.warpPerspective(img_bgr, M, (out_w, out_h))

def gather_by_subset(root):
    by = {}
    for p in root.rglob("*.jpg"):
        by.setdefault(discover_subset(p), []).append(p)
    return by

def stratified_sample(by_subset, max_images, min_per_subset, seed=0):
    rng = random.Random(seed)
    subsets = {k: list(v) for k,v in by_subset.items()}
    for v in subsets.values(): rng.shuffle(v)
    total = sum(len(v) for v in subsets.values())
    if max_images is None or max_images >= total:
        return subsets
    alloc = {k: min(min_per_subset, len(v)) for k,v in subsets.items()}
    used = sum(alloc.values()); remaining = max_images - used
    if remaining > 0:
        leftover = {k: len(v)-alloc[k] for k,v in subsets.items()}
        lt = sum(leftover.values())
        if lt > 0:
            for k in subsets:
                add = int(round(remaining * leftover[k] / lt))
                alloc[k] = min(len(subsets[k]), alloc[k]+add)
    over = sum(alloc.values()) - max_images
    if over > 0:
        for k in sorted(alloc, key=lambda z: alloc[z], reverse=True):
            cut = min(over, alloc[k]-min(min_per_subset, len(subsets[k])))
            alloc[k] -= max(cut,0); over -= max(cut,0)
            if over <= 0: break
    return {k: subsets[k][:alloc[k]] for k in subsets}

def stratified_test_split(sampled, test_frac, seed=0):
    rng = random.Random(seed+1)
    pool, test = {}, {}
    for k, v in sampled.items():
        v = list(v); rng.shuffle(v)
        nt = int(round(len(v)*test_frac))
        test[k] = v[:nt]; pool[k] = v[nt:]
    return pool, test

# Build the rectified-crop cache
CROPS_DIR  = WORK_ROOT / "crops"
LABELS_CSV = WORK_ROOT / "labels.csv"
CROPS_DIR.mkdir(parents=True, exist_ok=True)

def build_crop_cache():
    by = gather_by_subset(DATASET_ROOT)
    print("subsets found:", {k:len(v) for k,v in by.items()})
    sampled = stratified_sample(by, MAX_IMAGES, MIN_PER_SUBSET, seed=0)
    print("after sampling:", {k:len(v) for k,v in sampled.items()},
          "| total:", sum(len(v) for v in sampled.values()))
    pool, test = stratified_test_split(sampled, TEST_FRAC, seed=0)

    rows = []
    written = skipped = 0
    for split_name, split_dict in [("pool", pool), ("test", test)]:
        for subset, plist in split_dict.items():
            for p in plist:
                rec = parse_ccpd_filename(p)
                if rec is None: skipped += 1; continue
                img = cv2.imread(str(p))
                if img is None: skipped += 1; continue
                try:
                    crop = rectify_plate_bgr(img, rec["vertices"])
                except Exception:
                    skipped += 1; continue
                out = CROPS_DIR / f"{Path(p).stem}.jpg"
                cv2.imwrite(str(out), crop)
                rows.append({"path": str(out), "plate": rec["plate"],
                             "subset": subset, "split": split_name,
                             "src": str(p)})
                written += 1
        print(f"  {split_name}: {sum(len(v) for v in split_dict.values())} planned, written so far={written}")
    df = pd.DataFrame(rows)
    df.to_csv(LABELS_CSV, index=False)
    print(f"\nDone. {written} crops written, {skipped} skipped. Labels → {LABELS_CSV}")
    return df

if LABELS_CSV.exists():
    df_crops = pd.read_csv(LABELS_CSV)
    print(f"Loaded existing cache: {len(df_crops)} crops")
else:
    df_crops = build_crop_cache()

df_crops["plate_len"] = df_crops["plate"].str.len()
print("\nSplit distribution:"); print(df_crops.groupby(["split","subset"]).size().unstack(fill_value=0))

## 5. Visual sanity check of rectified crops

Before training, look at a few crops to confirm rectification is working. The plate should be **upright, centered, and fill the frame**. If crops come out rotated 90° or with characters running off the edge, the corner ordering is wrong — fix it before training.

In [ ]:
def preview_crops(df, n=8, seed=0):
    sample = df.sample(min(n, len(df)), random_state=seed)
    cols = 2; rows = math.ceil(len(sample)/cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols*6, rows*1.8))
    axes = np.array(axes).reshape(-1)
    for ax, (_, r) in zip(axes, sample.iterrows()):
        img = cv2.imread(r["path"])
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f'{r["plate"]}   [{r["subset"]} / {r["split"]}]', fontsize=11)
        ax.axis("off")
    for ax in axes[len(sample):]: ax.axis("off")
    plt.tight_layout(); plt.show()

preview_crops(df_crops, n=8)

## 6. PyTorch dataset

The dataset yields `(image_tensor, target_indices, target_length, plate_string)`. CTC's collate needs flat-concatenated targets plus per-sample lengths, so we use a custom `collate_fn`.

Normalization: we use simple `(x/255 - 0.5)/0.5` (range −1..1). Plenty for plate crops; no need for ImageNet stats since we're not using ImageNet pretraining.

In [ ]:
class PlateCropDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df = df.reset_index(drop=True)
        self.augment = augment
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = cv2.imread(r["path"])
        if img is None:
            # Fall back to a black image rather than crash; should never happen with cached crops
            img = np.zeros((CROP_H, CROP_W, 3), dtype=np.uint8)
        if img.shape[:2] != (CROP_H, CROP_W):
            img = cv2.resize(img, (CROP_W, CROP_H))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.augment:
            # mild augmentations: small color jitter + tiny translation
            if random.random() < 0.5:
                img = img.astype(np.float32) * random.uniform(0.85, 1.15)
                img = np.clip(img, 0, 255).astype(np.uint8)
            if random.random() < 0.5:
                tx = random.randint(-2, 2); ty = random.randint(-1, 1)
                M = np.float32([[1,0,tx],[0,1,ty]])
                img = cv2.warpAffine(img, M, (CROP_W, CROP_H), borderValue=(127,127,127))
        # to tensor, CHW, normalized to [-1, 1]
        x = (img.astype(np.float32) / 255.0 - 0.5) / 0.5
        x = torch.from_numpy(x).permute(2, 0, 1)  # 3 x H x W
        plate = r["plate"]
        idxs = torch.tensor(encode(plate), dtype=torch.long)
        return x, idxs, len(idxs), plate

def collate(batch):
    xs, targets, lengths, plates = zip(*batch)
    xs = torch.stack(xs, 0)                                  # B x 3 x H x W
    target_lengths = torch.tensor(lengths, dtype=torch.long) # B
    targets_flat = torch.cat(targets, 0)                     # sum(target_lengths)
    return xs, targets_flat, target_lengths, list(plates)

# quick sanity check
_ds = PlateCropDataset(df_crops.head(4))
_dl = DataLoader(_ds, batch_size=4, collate_fn=collate)
for xb, tb, tl, pb in _dl:
    print("batch image tensor:", xb.shape, "dtype:", xb.dtype)
    print("flat targets:", tb.shape, "lengths:", tl.tolist(), "plates:", pb)
    break

## 7. Model architecture — LPRNet (ported from `sirius-ai/LPRNet_Pytorch`)

After our previous CRNN-style attempt kept collapsing to all-blank predictions, we ported **LPRNet** — the architecture from Zherzdev & Gruzdev, *"LPRNet: License Plate Recognition via Deep Neural Networks"* (Intel, 2018). It's specifically designed for plates (not generic OCR), is widely used in production ALPR systems, and has well-documented results on Chinese license plates (~96% on the original repo's blue+green plate benchmark).

**Why LPRNet should work where our previous attempt didn't.** Three differences:

1. **Factorized convolution blocks** (`small_basic_block`) — instead of vanilla 3×3 convs, it uses a 1×1 → 3×1 → 1×3 → 1×1 sequence. This reduces parameters while keeping the receptive field, and the asymmetric (3×1, 1×3) factorization is particularly well-suited to the strongly horizontal layout of plate text.

2. **Multi-scale "global context"** — feature maps from 4 different depths (after ReLUs 2, 6, 13, 22) are power-normalized and concatenated. The model sees both low-level character strokes and high-level layout simultaneously, instead of relying only on the deepest features.

3. **Input is 24×94** (H×W), not 48×168. LPRNet was tuned for this size, and we resize crops to match. Smaller input also makes training much faster.

**For the CTC loss**, LPRNet produces an output of shape `(B, V, W)`; we permute to `(T, B, V)` to match `nn.CTCLoss`'s convention, exactly like before.

**Honest caveat.** The original sirius-ai code uses `nn.MaxPool3d` with a `(1,3,3)` kernel applied to 4D tensors — an unusual idiom that's hard to reproduce reliably across PyTorch versions. We use the equivalent `nn.MaxPool2d` with the corresponding spatial strides, which is what most LPRNet ports (e.g. `xuexingyu24/License_Plate_Detection_Pytorch`) do. Architecturally identical for our purposes; just cleaner.

In [ ]:
# === LPRNet — port of sirius-ai/LPRNet_Pytorch model/LPRNet.py ===
# Input size is 24×94 (H×W) — different from the previous 48×168 crops!
# The PlateCropDataset already resizes on the fly, so we just override the
# crop dimensions here. CROP_H and CROP_W will be re-bound below.

LPRNET_CROP_H, LPRNET_CROP_W = 24, 94

class small_basic_block(nn.Module):
    """Factorized basic block: 1×1 → 3×1 → 1×3 → 1×1. Cheaper than 3×3 with similar receptive field."""
    def __init__(self, ch_in, ch_out):
        super().__init__()
        ch_mid = ch_out // 4
        self.block = nn.Sequential(
            nn.Conv2d(ch_in,  ch_mid, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(ch_mid, ch_mid, kernel_size=(3, 1), padding=(1, 0)),
            nn.ReLU(),
            nn.Conv2d(ch_mid, ch_mid, kernel_size=(1, 3), padding=(0, 1)),
            nn.ReLU(),
            nn.Conv2d(ch_mid, ch_out, kernel_size=1),
        )
    def forward(self, x):
        return self.block(x)


class LPRNet(nn.Module):
    """LPRNet — Zherzdev & Gruzdev 2018, arXiv:1806.10447.

    Ported from sirius-ai/LPRNet_Pytorch with two minor changes:
      1. MaxPool3d → MaxPool2d (the original's MaxPool3d on 4D tensors is fragile
         across PyTorch versions; the spatial behavior is identical).
      2. Default class_num matches our CCPD vocabulary (68 = 67 chars + blank).

    Forward returns shape (T, B, V) ready for nn.CTCLoss.
    """
    def __init__(self, class_num=N_CLASSES, dropout_rate=0.5, lpr_max_len=8):
        super().__init__()
        self.class_num = class_num

        # Backbone — layer indices match the original code exactly (0..22)
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1),         # 0
            nn.BatchNorm2d(64),                                # 1
            nn.ReLU(),                                         # 2  ← global-ctx tap
            nn.MaxPool2d(kernel_size=3, stride=1),             # 3
            small_basic_block(64, 128),                        # 4
            nn.BatchNorm2d(128),                               # 5
            nn.ReLU(),                                         # 6  ← global-ctx tap
            nn.MaxPool2d(kernel_size=3, stride=(1, 2)),        # 7  (height/1, width/2)
            small_basic_block(128, 256),                       # 8   (note: 128 in, not 64 as in original — typo in original)
            nn.BatchNorm2d(256),                               # 9
            nn.ReLU(),                                         # 10
            small_basic_block(256, 256),                       # 11
            nn.BatchNorm2d(256),                               # 12
            nn.ReLU(),                                         # 13 ← global-ctx tap
            nn.MaxPool2d(kernel_size=3, stride=(1, 2)),        # 14 (height/1, width/2)
            nn.Dropout(dropout_rate),                          # 15
            nn.Conv2d(256, 256, kernel_size=(1, 4), stride=1), # 16
            nn.BatchNorm2d(256),                               # 17
            nn.ReLU(),                                         # 18
            nn.Dropout(dropout_rate),                          # 19
            nn.Conv2d(256, class_num, kernel_size=(13, 1)),    # 20  (height/13, width/1)
            nn.BatchNorm2d(class_num),                         # 21
            nn.ReLU(),                                         # 22 ← global-ctx tap
        )
        # Container — fuses concatenated global-context features → class_num
        # Concat channel count: 64 + 128 + 256 + class_num
        self.container = nn.Conv2d(64 + 128 + 256 + class_num, class_num, kernel_size=1)

        # Init: bias the blank logit negative — keeps the all-blank attractor weaker.
        with torch.no_grad():
            self.container.bias.zero_()
            self.container.bias[BLANK_IDX] = -2.0
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None and m is not self.container:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, x):
        # Collect global-context features from layers 2, 6, 13, 22
        keep_layers = {2, 6, 13, 22}
        feats = []
        for i, layer in enumerate(self.backbone):
            x = layer(x)
            if i in keep_layers:
                feats.append(x)
        # Power-normalize each captured feature: f / mean(f**2).
        # Then resize each to match the final feature map's H×W, concat along channels.
        final = feats[-1]
        H_out, W_out = final.shape[2], final.shape[3]
        normed = []
        for f in feats:
            f_pow = f.pow(2)
            f_mean = f_pow.mean()
            f_norm = f.div(f_mean.clamp(min=1e-8))
            if f.shape[2:] != (H_out, W_out):
                f_norm = F.adaptive_avg_pool2d(f_norm, (H_out, W_out))
            normed.append(f_norm)
        cat = torch.cat(normed, dim=1)                    # B x (64+128+256+class_num) x H_out x W_out
        logits = self.container(cat)                       # B x class_num x H_out x W_out
        # Average along the height axis → sequence of length W_out
        logits = logits.mean(dim=2)                        # B x class_num x W_out
        logits = logits.permute(2, 0, 1).contiguous()      # T x B x V  (ready for CTCLoss)
        return logits


# Keep `CRNN` alias so the rest of the notebook (training loop, CV, eval) is unchanged
CRNN = LPRNet

# Resize the crops to LPRNet's native input size — most important detail.
CROP_H, CROP_W = LPRNET_CROP_H, LPRNET_CROP_W
print(f"Crops will be resized to {CROP_H}×{CROP_W} (H×W) for LPRNet")

# Sanity: shape and parameter count
_m = LPRNet(N_CLASSES)
_x = torch.randn(2, 3, CROP_H, CROP_W)
with torch.no_grad():
    _y = _m(_x)
print("model output:", _y.shape, "(expect T x 2 x", N_CLASSES, ")")
print(f"params: {sum(p.numel() for p in _m.parameters())/1e6:.2f} M  (LPRNet paper reports ~1.7 MB / ~0.5 M params)")


## 8. Greedy CTC decoder + accuracy metrics

**Greedy CTC decoding**: at each time-step, take the argmax. Then collapse consecutive duplicates and drop blanks. This is the standard fast decoder — beam search can do a bit better but greedy is fine for evaluating a model.

**Metrics**:
- **Full-plate accuracy**: 1 if the decoded string exactly matches the ground truth, else 0. This is the project's all-or-nothing metric.
- **Per-position accuracy**: requires the decoded string to be length 7; if it is, we score each position independently. If it isn't length 7, all 7 positions count as wrong for this image (strict, but honest).
- **Mean edit distance**: how many character edits (insert/delete/substitute) separate prediction from truth. A "softer" diagnostic.

In [ ]:
def greedy_ctc_decode(logits_TBV):
    '''logits: T x B x V (raw or log-softmax — argmax is the same). Returns list[str].'''
    # argmax along V → T x B; then we go per-batch and collapse
    idx = logits_TBV.argmax(dim=2).cpu().numpy()      # T x B
    T, B = idx.shape
    out = []
    for b in range(B):
        seq = []
        prev = -1
        for t in range(T):
            c = int(idx[t, b])
            if c != prev and c != BLANK_IDX:
                seq.append(IDX2CHAR[c])
            prev = c
        out.append(''.join(seq))
    return out

def edit_distance(a: str, b: str) -> int:
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    dp = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        prev, dp[0] = dp[0], i
        for j, cb in enumerate(b, 1):
            cur = dp[j]
            dp[j] = prev if ca == cb else 1 + min(prev, dp[j], dp[j-1])
            prev = cur
    return dp[-1]

def evaluate(model, loader, device):
    model.eval()
    full_correct = 0; total = 0; ed_sum = 0
    pos_correct = np.zeros(7, dtype=np.int64); pos_total = np.zeros(7, dtype=np.int64)
    with torch.no_grad():
        for xb, _, _, plates in loader:
            xb = xb.to(device)
            logits = model(xb)                              # T x B x V
            preds = greedy_ctc_decode(logits)
            for pred, truth in zip(preds, plates):
                total += 1
                ed_sum += edit_distance(pred, truth)
                if pred == truth: full_correct += 1
                if len(truth) == 7:
                    for i in range(7):
                        pos_total[i] += 1
                        if i < len(pred) and pred[i] == truth[i]:
                            pos_correct[i] += 1
    return {
        "full_plate_acc": full_correct / max(total, 1),
        "mean_edit_dist": ed_sum / max(total, 1),
        "per_position_acc": (pos_correct / np.maximum(pos_total, 1)).tolist(),
        "n": total,
    }

# tiny sanity test of decoder
_logits = torch.full((10, 1, N_CLASSES), -10.0)
# craft a sequence: [BLANK, BLANK, '皖', '皖', BLANK, 'A', BLANK, '1', BLANK, BLANK]
seq = [BLANK_IDX, BLANK_IDX, CHAR2IDX['皖'], CHAR2IDX['皖'], BLANK_IDX,
       CHAR2IDX['A'], BLANK_IDX, CHAR2IDX['1'], BLANK_IDX, BLANK_IDX]
for t, c in enumerate(seq):
    _logits[t, 0, c] = 10.0
print("decoder test → expect '皖A1':", greedy_ctc_decode(_logits))

## 9. Training loop for one fold

One-fold training: build train and val loaders, train with CTC loss, early-stop on **validation edit distance** (lower is better).

Notes on the loss and training dynamics:
- `nn.CTCLoss` expects `log_probs` of shape `(T, B, V)`, targets flat-concatenated, plus `input_lengths` (all = T) and `target_lengths`.
- `zero_infinity=True` is important — without it, a temporarily-bad batch can produce ∞ loss and corrupt training.
- **Why edit distance for early stopping, not full-plate accuracy?** Early in CTC training the model often predicts very short strings for many epochs while it learns alignment. Full-plate accuracy stays at exactly 0 the whole time and gives no useful gradient for early-stopping; edit distance falls smoothly and is the right signal.
- **Blank-collapse mitigation**: the model initialization biases the final classifier's blank logit to −2.0. Combined with the pure-conv architecture (no LSTM), this avoids the all-blank attractor that caused our previous CRNN to stall.

In [ ]:
def train_one_fold(train_df, val_df, epochs=30, batch_size=128, lr=1e-3,
                   weight_decay=1e-4, patience=20, augment=True, verbose=True,
                   tag="fold"):
    train_loader = DataLoader(PlateCropDataset(train_df, augment=augment),
                              batch_size=batch_size, shuffle=True, num_workers=0,
                              collate_fn=collate, pin_memory=(DEVICE=="cuda"))
    val_loader   = DataLoader(PlateCropDataset(val_df, augment=False),
                              batch_size=batch_size, shuffle=False, num_workers=0,
                              collate_fn=collate, pin_memory=(DEVICE=="cuda"))

    model = CRNN(N_CLASSES).to(DEVICE)
    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    ctc = nn.CTCLoss(blank=BLANK_IDX, zero_infinity=True)

    # Track best by EDIT DISTANCE (lower is better). Full-plate stays 0 for many
    # epochs during normal CTC training, so it's a poor early-stop signal.
    best_edit = float("inf"); best_acc = 0.0; best_state = None; stale = 0
    history = []
    for ep in range(1, epochs+1):
        model.train()
        loss_sum = 0.0; n = 0
        t0 = time.time()
        for xb, targets, target_lens, _ in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            targets = targets.to(DEVICE)
            target_lens = target_lens.to(DEVICE)
            logits = model(xb)                                       # T x B x V
            log_probs = F.log_softmax(logits, dim=2)
            T_, B, _ = log_probs.shape
            input_lens = torch.full((B,), T_, dtype=torch.long, device=DEVICE)
            loss = ctc(log_probs, targets, input_lens, target_lens)
            optim.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optim.step()
            loss_sum += loss.item() * B; n += B
        sched.step()
        train_loss = loss_sum / max(n, 1)

        val = evaluate(model, val_loader, DEVICE)
        history.append({"epoch": ep, "train_loss": train_loss,
                        "val_full_acc": val["full_plate_acc"],
                        "val_edit": val["mean_edit_dist"]})
        if verbose:
            dt = time.time() - t0
            print(f"  ep {ep:3d} | loss {train_loss:.4f} | "
                  f"val full-plate {val['full_plate_acc']:.4f} | "
                  f"edit {val['mean_edit_dist']:.3f} | {dt:.1f}s")
        if val["mean_edit_dist"] < best_edit:
            best_edit = val["mean_edit_dist"]
            best_acc  = val["full_plate_acc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                if verbose: print(f"  early stop @ epoch {ep} "
                                  f"(best edit={best_edit:.3f}, full-plate={best_acc:.4f})")
                break

    # restore best weights for evaluation
    model.load_state_dict(best_state)
    return model, best_acc, pd.DataFrame(history)

## 10. Stratified 4-fold cross-validation

Same fold logic as the pose notebook: balance subsets across folds via round-robin within each subset. CV trains 4 models, each on 3 folds and validated on 1, then averages.

In [ ]:
def make_stratified_folds(df, n_folds, seed=0):
    rng = random.Random(seed + 7)
    by_subset = {}
    for i, r in df.iterrows():
        by_subset.setdefault(r["subset"], []).append(i)
    fold_of = {}
    for subset, ids in by_subset.items():
        ids = list(ids); rng.shuffle(ids)
        for j, idx in enumerate(ids):
            fold_of[idx] = j % n_folds
    df = df.copy()
    df["fold"] = df.index.map(fold_of)
    return df

pool_df = df_crops[df_crops["split"]=="pool"].reset_index(drop=True)
test_df = df_crops[df_crops["split"]=="test"].reset_index(drop=True)
pool_df = make_stratified_folds(pool_df, N_FOLDS, seed=0)

# sanity: balance check
print("pool size:", len(pool_df), "| test size:", len(test_df))
print("\nfold sizes:", pool_df["fold"].value_counts().sort_index().to_dict())
print("\nsubset × fold (pool):")
print(pool_df.groupby(["subset","fold"]).size().unstack(fill_value=0))

In [ ]:
def run_cross_validation(pool_df, epochs=30, batch_size=128, lr=1e-3, verbose=True):
    results = []
    for k in range(N_FOLDS):
        if verbose: print(f"\n=== fold {k} ===")
        train_df = pool_df[pool_df["fold"] != k]
        val_df   = pool_df[pool_df["fold"] == k]
        if verbose: print(f"  train: {len(train_df)} | val: {len(val_df)}")
        model, best_acc, hist = train_one_fold(
            train_df, val_df,
            epochs=epochs, batch_size=batch_size, lr=lr,
            tag=f"fold{k}", verbose=verbose
        )
        # full val evaluation at best epoch
        val_loader = DataLoader(PlateCropDataset(val_df), batch_size=batch_size,
                                shuffle=False, num_workers=0, collate_fn=collate)
        metrics = evaluate(model, val_loader, DEVICE)
        metrics["fold"] = k
        results.append(metrics)
        del model; torch.cuda.empty_cache() if DEVICE == "cuda" else None
    return pd.DataFrame(results)

# Run 4-fold CV. Adjust EPOCHS_CV based on how long the first fold takes.
EPOCHS_CV   = 30
BATCH_SIZE  = 128

cv_results = run_cross_validation(pool_df, epochs=EPOCHS_CV, batch_size=BATCH_SIZE)
display(cv_results)

print("\n=== Cross-validated summary ===")
print(f"Full-plate accuracy: {cv_results['full_plate_acc'].mean():.4f} ± {cv_results['full_plate_acc'].std():.4f}")
print(f"Mean edit distance : {cv_results['mean_edit_dist'].mean():.3f} ± {cv_results['mean_edit_dist'].std():.3f}")

# per-position accuracy averaged across folds
pp = np.array(cv_results["per_position_acc"].tolist())   # K x 7
print("\nPer-position accuracy (mean ± std across folds):")
pos_labels = ["province","letter","alnum3","alnum4","alnum5","alnum6","alnum7"]
for i, lab in enumerate(pos_labels):
    print(f"  pos {i+1} ({lab:8s}): {pp[:,i].mean():.4f} ± {pp[:,i].std():.4f}")

In [ ]:
xb, _, _, plates = next(iter(val_loader))
with torch.no_grad():
    logits = model(xb.to(DEVICE))
    probs = F.softmax(logits, dim=2)

print(f"plate: {plates[0]}")
top_idx = logits[:, 0, :].argmax(dim=1).cpu().numpy()
print(f"argmax per timestep: {top_idx.tolist()}")
print(f"unique classes predicted: {sorted(set(top_idx.tolist()))}")
print(f"mean P(blank): {probs[:, 0, BLANK_IDX].mean().item():.3f}")

# Decode a few plates
preds = greedy_ctc_decode(logits)
for truth, pred in zip(plates[:5], preds[:5]):
    print(f"truth: {truth!r:12s}  pred: {pred!r}")

## 11. Retrain on the full pool and evaluate on the frozen test set

After CV has measured how this configuration generalizes, train one final model on all pool data (no fold held out) and evaluate **once** on the frozen test set. This is the headline number for the paper.

In [ ]:
FINAL_EPOCHS = 40

# Use 10% of the pool as a held-out val for early stopping during the final retrain.
# This is NOT the frozen test set — we still don't touch that until the next cell.
final_train, final_val = [], []
rng = random.Random(123)
for subset, group in pool_df.groupby("subset"):
    idxs = list(group.index); rng.shuffle(idxs)
    nv = max(1, int(0.10 * len(idxs)))
    final_val.extend(idxs[:nv])
    final_train.extend(idxs[nv:])
ft_df = pool_df.loc[final_train].reset_index(drop=True)
fv_df = pool_df.loc[final_val].reset_index(drop=True)
print(f"final retrain — train: {len(ft_df)} | val: {len(fv_df)}")

final_model, final_best_val, final_hist = train_one_fold(
    ft_df, fv_df, epochs=FINAL_EPOCHS, batch_size=BATCH_SIZE,
    lr=1e-3, patience=20, tag="final", verbose=True
)
print(f"\nFinal model best val full-plate accuracy: {final_best_val:.4f}")

In [ ]:
# === FROZEN TEST EVALUATION ===
test_loader = DataLoader(PlateCropDataset(test_df), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=0, collate_fn=collate)
test_metrics = evaluate(final_model, test_loader, DEVICE)

print("=" * 60)
print("FROZEN TEST-SET RESULTS")
print("=" * 60)
print(f"Full-plate accuracy: {test_metrics['full_plate_acc']:.4f}")
print(f"Mean edit distance : {test_metrics['mean_edit_dist']:.3f}")
print(f"Test images        : {test_metrics['n']}")

print("\nPer-position accuracy:")
for i, (lab, acc) in enumerate(zip(pos_labels, test_metrics["per_position_acc"])):
    print(f"  pos {i+1} ({lab:8s}): {acc:.4f}")

# Save the final model
ckpt = WORK_ROOT / "crnn_final.pt"
torch.save({"model_state_dict": final_model.state_dict(),
            "vocab": VOCAB,
            "crop_size": (CROP_H, CROP_W),
            "test_metrics": test_metrics},
           ckpt)
print(f"\nSaved final model → {ckpt}")

## 12. Per-subset test breakdown (for the paper's results table)

Break down test accuracy by CCPD subset (Base / Rotate / Tilt / …). This is the table reviewers expect to see — it directly answers "how does the model fare in harder conditions?".

In [ ]:
rows = []
for subset, group in test_df.groupby("subset"):
    loader = DataLoader(PlateCropDataset(group), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=0, collate_fn=collate)
    m = evaluate(final_model, loader, DEVICE)
    rows.append({
        "subset": subset,
        "n": m["n"],
        "full_plate_acc": m["full_plate_acc"],
        "mean_edit_dist": m["mean_edit_dist"],
    })
per_subset = pd.DataFrame(rows).sort_values("subset")
display(per_subset)

## 13. Sample predictions on the test set

Show a few good and bad predictions to get a qualitative feel for failure modes.

In [ ]:
def show_predictions(model, df, n=8, seed=0):
    sample = df.sample(min(n, len(df)), random_state=seed).reset_index(drop=True)
    loader = DataLoader(PlateCropDataset(sample), batch_size=n, shuffle=False,
                        collate_fn=collate)
    model.eval()
    with torch.no_grad():
        for xb, _, _, plates in loader:
            logits = model(xb.to(DEVICE))
            preds = greedy_ctc_decode(logits)
            break
    cols = 2; rows = math.ceil(len(sample)/cols)
    fig, axes = plt.subplots(rows, cols, figsize=(cols*6, rows*1.8))
    axes = np.array(axes).reshape(-1)
    for ax, (_, r), pred in zip(axes, sample.iterrows(), preds):
        img = cv2.imread(r["path"])
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ok = "✓" if pred == r["plate"] else "✗"
        ax.set_title(f'{ok}  truth: {r["plate"]}   pred: {pred}', fontsize=10)
        ax.axis("off")
    for ax in axes[len(sample):]: ax.axis("off")
    plt.tight_layout(); plt.show()

show_predictions(final_model, test_df, n=8)

## 14. End-to-end check: CRNN on YOLO-pose-predicted corners

Optional but important for the paper. So far, training and testing used **ground-truth** corners — that's an upper bound on real performance. To estimate the actual end-to-end accuracy when chained with your YOLO-pose model, re-rectify each test image from **predicted** corners and re-evaluate.

To run this you need (a) your trained YOLO-pose model weights and (b) the original CCPD images (not the cached crops). If you don't have those handy in this kernel, skip this section for now and run it from the pose notebook.

In [ ]:
# This is a sketch — uncomment and edit paths to enable.
# 
# from ultralytics import YOLO
# pose_model_path = "runs_plate_pose/final_model/weights/best.pt"
# yolo_pose = YOLO(pose_model_path)
# 
# rows = []
# for _, r in test_df.iterrows():
#     src = Path(r["src"])  # original full image path saved during cache build
#     img = cv2.imread(str(src))
#     if img is None: continue
#     pred = yolo_pose.predict(str(src), verbose=False)[0]
#     if pred.keypoints is None or len(pred.keypoints) == 0: continue
#     kp = pred.keypoints.xy.cpu().numpy()[0]    # (4, 2) — TL, TR, BR, BL
#     crop = rectify_plate_bgr(img, kp.tolist())
#     # run through CRNN
#     x = (cv2.cvtColor(crop, cv2.COLOR_BGR2RGB).astype(np.float32)/255.0 - 0.5) / 0.5
#     x = torch.from_numpy(x).permute(2,0,1).unsqueeze(0).to(DEVICE)
#     with torch.no_grad():
#         logits = final_model(x)
#     pred_str = greedy_ctc_decode(logits)[0]
#     rows.append({"truth": r["plate"], "pred_e2e": pred_str,
#                  "match": pred_str == r["plate"], "subset": r["subset"]})
# 
# e2e = pd.DataFrame(rows)
# print(f"End-to-end full-plate accuracy: {e2e['match'].mean():.4f} on {len(e2e)} images")
# display(e2e.groupby("subset")["match"].agg(["mean", "count"]))
print("End-to-end cell is a sketch — see comments above.")

## 15. Next steps & notes

**For the paper.** Report two complementary tables:
- *CV table* showing 4-fold mean ± std for the chosen configuration → demonstrates methodology rigor.
- *Frozen test-set result*, broken down per CCPD subset → the headline number.

If you want a model comparison (the project rewards comparing variants), the natural axis here is **model capacity via the CFG list**. Easy variations to try without redesigning anything:
- `CFG_SMALL` (~0.3M params, fastest) vs `CFG_MEDIUM` (~1.3M) vs `CFG_BIG` (~5M, the LPDet-equivalent).
- Augmentation on/off.
- Crop size: 32×128 vs 48×168 vs 64×224.

To switch CFG: replace `CRNN(N_CLASSES)` with `CRNN(N_CLASSES, cfg=CFG_BIG)` in the training loop.

**Honest caveats**
- The corner-ordering step is still the silent killer — if section 5's crop preview shows rotated/mirrored plates, fix `order_corners` before anything else.
- LPDet's CRNN_Tiny reported ~76% accuracy on the full CCPD test set — that's a sobering reference point for what's achievable on this dataset at small model scale.
- This architecture was ported (in spirit) from `we0091234/crnn_plate_recognition` rather than implemented from scratch by us — the upstream is known to work on CCPD, so structural pathologies should be much less likely than with our previous LSTM-based attempt.
- The model output is `T x B x V` and CTC takes `(T, B, V)` log-probs — conventions match the previous code, so the training loop is unchanged.